In [12]:
#WEEK 3 8-PUZZLE using A*
import heapq

class Node:
    def __init__(self, board, g, h, parent=None, move=""):
        self.board = board  # 3x3 list representing the board
        self.g = g          # Cost from start to current node (depth)
        self.h = h          # Heuristic cost (Manhattan distance to goal)
        self.f = g + h      # f(n) = g(n) + h(n)
        self.parent = parent
        self.move = move

    # Priority queue comparison based on f(n)
    def __lt__(self, other):
        return self.f < other.f

def get_blank_position(board):
    for i in range(3):
        for j in range(3):
            if board[i][j] == 0:
                return i, j
    return -1, -1

# Calculate Manhattan Distance Heuristic
def calculate_manhattan(board, goal):
    distance = 0
    goal_pos = {}
    for i in range(3):
        for j in range(3):
            goal_pos[goal[i][j]] = (i, j)

    for i in range(3):
        for j in range(3):
            val = board[i][j]
            if val != 0:
                target_x, target_y = goal_pos[val]
                distance += abs(i - target_x) + abs(j - target_y)
    return distance

def get_heuristic(board, goal):
    return calculate_manhattan(board, goal)

def get_neighbors(node, goal):
    neighbors = []
    x, y = get_blank_position(node.board)

    # Possible movements: Up, Down, Left, Right
    directions = [
        (-1, 0, "Up"),
        (1, 0, "Down"),
        (0, -1, "Left"),
        (0, 1, "Right")
    ]

    for dx, dy, move in directions:
        new_x, new_y = x + dx, y + dy
        if 0 <= new_x < 3 and 0 <= new_y < 3:
            # Create a new board configuration by swapping
            new_board = [row[:] for row in node.board]
            new_board[x][y], new_board[new_x][new_y] = new_board[new_x][new_y], new_board[x][y]

            g_val = node.g + 1
            h_val = get_heuristic(new_board, goal)
            neighbors.append(Node(new_board, g_val, h_val, node, move))

    return neighbors

def solve_8_puzzle(initial_board, goal_board):
    open_set = []
    closed_set = set()

    h_init = get_heuristic(initial_board, goal_board)
    root = Node(initial_board, 0, h_init)
    heapq.heappush(open_set, root)

    print("\n--- Starting A* Search ---")

    while open_set:
        current = heapq.heappop(open_set)

        # Print current evaluation metrics f(n) = g(n) + h(n)
        print(f"Expanding State with f(n) = {current.f} (g(n) = {current.g}, h(n) = {current.h}) Move: {current.move or 'Start'}")
        for row in current.board:
            print("  ", row)
        print()

        # Check if goal reached
        if current.board == goal_board:
            path = []
            temp = current
            while temp.parent:
                path.append((temp.move, temp.board))
                temp = temp.parent
            path.reverse()
            return path, current.g

        closed_set.add(tuple(tuple(row) for row in current.board))

        for neighbor in get_neighbors(current, goal_board):
            neighbor_tuple = tuple(tuple(row) for row in neighbor.board)
            if neighbor_tuple in closed_set:
                continue
            heapq.heappush(open_set, neighbor)

    return None, -1

# Function to take valid 3x3 matrix input from user
def read_board(prompt):
    while True:
        print(prompt)
        board = []
        flat_board = []
        try:
            for i in range(3):
                row = list(map(int, input(f"Row {i+1} (space-separated, use 0 for blank): ").strip().split()))
                if len(row) != 3:
                    raise ValueError("Each row must have exactly 3 numbers.")
                board.append(row)
                flat_board.extend(row)
            if sorted(flat_board) != list(range(9)):
                print("Invalid board! Must contain digits 0 to 8 without duplicates. Please try again.\n")
                continue
            return board
        except ValueError as e:
            print(f"Invalid input: {e} Please enter integers only and complete all rows.\n")

if __name__ == "__main__":
    print("Bhashitha G S")
    print("1WN24CS062")
    print("Enter the Initial State (0-8):")
    initial = read_board("Initial Board Configuration:")

    print("\nEnter the Goal State (0-8):")
    goal = read_board("Goal Board Configuration:")

    solution_path, total_moves = solve_8_puzzle(initial, goal)

    if solution_path:
        print(f"\nSolution found in {total_moves} moves!")
        print("Steps:")
        step_num = 1
        for move, board in solution_path:
            print(f"Step {step_num}: Move {move}")
            for row in board:
                print("  ", row)
            print()
            step_num += 1
    else:
        print("No solution found.")

Bhashitha G S
1WN24CS062
Enter the Initial State (0-8):
Initial Board Configuration:
Row 1 (space-separated, use 0 for blank): 2 8 3
Row 2 (space-separated, use 0 for blank): 1 6 4
Row 3 (space-separated, use 0 for blank): 0 7 5

Enter the Goal State (0-8):
Goal Board Configuration:
Row 1 (space-separated, use 0 for blank): 1  2 3
Row 2 (space-separated, use 0 for blank): 8 0 4
Row 3 (space-separated, use 0 for blank): 7 6 5

--- Starting A* Search ---
Expanding State with f(n) = 6 (g(n) = 0, h(n) = 6) Move: Start
   [2, 8, 3]
   [1, 6, 4]
   [0, 7, 5]

Expanding State with f(n) = 6 (g(n) = 1, h(n) = 5) Move: Right
   [2, 8, 3]
   [1, 6, 4]
   [7, 0, 5]

Expanding State with f(n) = 6 (g(n) = 2, h(n) = 4) Move: Up
   [2, 8, 3]
   [1, 0, 4]
   [7, 6, 5]

Expanding State with f(n) = 6 (g(n) = 3, h(n) = 3) Move: Up
   [2, 0, 3]
   [1, 8, 4]
   [7, 6, 5]

Expanding State with f(n) = 6 (g(n) = 4, h(n) = 2) Move: Left
   [0, 2, 3]
   [1, 8, 4]
   [7, 6, 5]

Expanding State with f(n) = 6 (g(n)

In [7]:
#WEEK 3 8-PUZZLE using IDFS
class Node:
    def __init__(self, board, parent=None, move=None, depth=0):
        self.board = board
        self.parent = parent
        self.move = move
        self.depth = depth

    @property
    def state_tuple(self):
        return tuple(self.board)

    def get_successors(self):
        successors = []
        zero_idx = self.board.index(0)
        row, col = divmod(zero_idx, 3)
        moves = {'Up': -3, 'Down': 3, 'Left': -1, 'Right': 1}

        for move_name, delta in moves.items():
            new_row, new_col = divmod(zero_idx + delta, 3)
            if 0 <= new_row < 3 and 0 <= new_col < 3 and abs(row - new_row) + abs(col - new_col) == 1:
                new_board = self.board[:]
                new_board[zero_idx], new_board[zero_idx + delta] = new_board[zero_idx + delta], new_board[zero_idx]
                successors.append(Node(new_board, parent=self, move=move_name, depth=self.depth + 1))
        return successors

def print_board(board):
    for i in range(0, 9, 3):
        print(" ".join(str(x) if x != 0 else '_' for x in board[i:i+3]))
    print()

def depth_limited_search(node, goal_state, limit, visited):
    if node.board == goal_state:
        return node
    if node.depth == limit:
        return None
    visited.add(node.state_tuple)
    for succ in node.get_successors():
        if succ.state_tuple not in visited:
            result = depth_limited_search(succ, goal_state, limit, visited)
            if result is not None:
                return result
    visited.remove(node.state_tuple)
    return None

def iddfs(initial_state, goal_state, max_depth=30):
    for limit in range(max_depth + 1):
        visited = set()
        result = depth_limited_search(Node(initial_state), goal_state, limit, visited)
        if result is not None:
            return result, limit
    return None, max_depth

def parse_input(prompt):
    while True:
        try:
            val = input(prompt).strip().split()
            if len(val) != 9:
                print("Please enter exactly 9 numbers separated by spaces.")
                continue
            board = [int(x) for x in val]
            if sorted(board) != list(range(9)):
                print("Invalid input! Please enter numbers from 0 to 8 without duplicates.")
                continue
            return board
        except ValueError:
            print("Invalid input! Please enter integers only.")

if __name__ == "__main__":
    print("Bhashitha G S")
    print("1WN24CS062")
    initial_state = parse_input("Enter initial state: ")
    goal_state = parse_input("Enter goal state:   ")
    solution_node, depth_reached = iddfs(initial_state, goal_state)

    if solution_node:
        path = []
        curr = solution_node
        while curr:
            path.append(curr)
            curr = curr.parent
        path.reverse()
        for step_num, node in enumerate(path):
            print(f"Step {step_num}:" + (f" Move {node.move}" if node.move else " Initial State"))
            print_board(node.board)
    else:
        print("No solution found within the depth limit.")


Bhashitha G S
1WN24CS062
Enter initial state: 2 8 3 1 6 4 0 7 5
Enter goal state:   1 2 3 8 0 4 7 5
Please enter exactly 9 numbers separated by spaces.
Enter goal state:   1 2 3 8 0 4 7 6 5
Step 0: Initial State
2 8 3
1 6 4
_ 7 5

Step 1: Move Right
2 8 3
1 6 4
7 _ 5

Step 2: Move Up
2 8 3
1 _ 4
7 6 5

Step 3: Move Up
2 _ 3
1 8 4
7 6 5

Step 4: Move Left
_ 2 3
1 8 4
7 6 5

Step 5: Move Down
1 2 3
_ 8 4
7 6 5

Step 6: Move Right
1 2 3
8 _ 4
7 6 5

